> `oeai_mod_bromcom_gold` - Gold notebook for Bromcom

In [ ]:
%run oeai_mod_bromcom_env_var

In [ ]:
# Initialise Logging
oeai.notebook = SimpleNamespace(
    name = "oeai_mod_bromcom_gold.ipynb",
    buildversion = "20251105.1",
    buildtimestamp = "2025-11-05T16:00:00Z",
)
oeai.log.init()
oeai.log.start_block("Notebook Init")

In [ ]:
from pyspark.sql import DataFrame
from typing import Tuple
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType, StringType, ArrayType, StructType, StructField, DateType
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, month, dayofmonth
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

In [ ]:
# List of Delta table names to process
delta_tables = ["dim_Organisation", 
                "dim_Date",
                "dim_Student", 
                "dim_StudentExtended", 
                "fact_AttendanceSummary", 
                "fact_AttendanceSession", 
                "fact_Exclusion",
                "fact_Behaviour",
                "fact_Achievement",
                "dim_ExclusionReason",
                # "fact_Attainment",
                "dim_Staff",
                "fact_StaffAbsence",
                # "fact_AttendanceLesson",
                ]

# Functions

##### These functions are all used in the processing block

In [ ]:
from pyspark.sql import DataFrame, functions as F, types as T

def ensure_academic_year_column(
    df: DataFrame,
    date_column: str,
    input_format: str = "yyyy-MM-dd",
    start_month: int | None = None,
    start_day: int | None = None,
    output_col: str = "Academic_Year",
) -> DataFrame:
    """
    Adds 'Academic_Year' (e.g. '2024/2025') using a configurable month+day
    boundary for the academic year start. If start_month/day are not provided,
    they are taken from ACADEMIC_YEAR_START.
    """
    if date_column not in df.columns:
        raise ValueError(f"Column '{date_column}' not found in DataFrame.")

    # Resolve the effective start month/day
    sm = start_month if start_month is not None else ACADEMIC_YEAR_START["month"]
    sd = start_day if start_day is not None else ACADEMIC_YEAR_START["day"]

    # Cast to DateType if needed
    df = df.withColumn(date_column, F.to_date(F.col(date_column), input_format))

    # Build boundary date for each row's calendar year: YYYY-sm-sd
    year_col = F.year(F.col(date_column))
    # Prefer make_date if available (Spark 2.4+). If your cluster is older, see note below.
    boundary = F.make_date(year_col, F.lit(sm), F.lit(sd))

    # Determine start_year based on whether the row's date is on/after the boundary
    start_year = F.when(F.col(date_column) >= boundary, year_col).otherwise(year_col - F.lit(1))
    end_year = start_year + F.lit(1)

    academic_year = F.concat_ws("/", start_year.cast(T.StringType()), end_year.cast(T.StringType()))
    return df.withColumn(output_col, academic_year)


In [ ]:
# Function to generate student_ext_academic_year

from pyspark.sql import DataFrame

def add_student_ext_academic_year_key(df: DataFrame) -> DataFrame:
    try:
        _fn ="add_student_ext_academic_year_key"

        # Load dim_StudentExtendedAcademicYear with only the necessary columns
        dim_student_academic_year_group = spark.read.parquet(f"{gold_path}/dim_StudentExtendedAcademicYear").select(
            "studentkey", "Academic_Year", "studentextendedacademicyearkey"
        )
        # Check if the required columns exist in the input DataFrame
        if "studentkey" not in df.columns or "Academic_Year" not in df.columns:
            raise ValueError("The DataFrame must contain both 'studentkey' and 'Academic_Year' columns for the join.")

        # Join the fact table with dim_StudentExtendedAcademicYear on studentkey and Academic_Year
        df_with_key = df.join(
            dim_student_academic_year_group,
            on=["studentkey", "Academic_Year"],
            how="left"
        )

        return df_with_key

    except ValueError as ve:
        # print(f"ValueError: {ve}")
        oeai.log.exception(f"ERROR - {_fn}", function=_fn)
        raise  # Re-raise the exception after logging

    except Exception as e:
        # print(f"An error occurred while adding student academic year group key: {e}")
        oeai.log.exception(f"ERROR - {_fn}", function=_fn)
        raise  # Re-raise the exception after logging

In [ ]:
def add_student_academic_year_group_key(df: DataFrame) -> DataFrame:
    # Load dim_StudentAcademicYearGroup with only the necessary columns
    dim_student_academic_year_group = spark.read.parquet(f"{gold_path}/dim_StudentAcademicYearGroup").select(
        "studentkey", "Academic_Year", "studentacademicyeargroupkey"
    )

    # Join the fact table with dim_StudentAcademicYearGroup on studentkey and Academic_Year
    df_with_key = df.join(
        dim_student_academic_year_group,
        on=["studentkey", "Academic_Year"],
        how="left"
    )

    return df_with_key

In [ ]:
# Function to map age to year group
def get_year_group_from_age(age):
    year_group_mapping = {
        0: -4, 1: -3, 2: -2, 3: -1, 4: 0,
        5: 1, 6: 2, 7: 3, 8: 4, 9: 5,
        10: 6, 11: 7, 12: 8, 13: 9, 14: 10,
        15: 11, 16: 12, 17: 13, 18: 14
    }
    return year_group_mapping.get(age, None)

In [ ]:
from datetime import datetime, date

def generate_academic_years_and_groups(Current_YG, current_academic_year, leaving_date, years_back=4):
    """
    Returns a list of tuples: (academic_year_str, 'Year N', mapped_label)

    Assumptions:
    - Current_YG is accurate for the given current_academic_year as is based on either Bromcom data or calculated from DoB.
    - Build rows for the current year and the previous (years_back-1) years by
      decrementing YG relative to the current academic year.
    - Then remove rows where the academic year start (Aug 1) is AFTER the leaving_date.
      If leaving_date is None, no rows are removed.
    """
    if Current_YG is None or not current_academic_year:
        return []

    # Parse "YYYY/ZZZZ" to get the start year YYYY
    start_year = int(str(current_academic_year).split('/')[0])

    # Normalize leaving_date -> date | None
    if isinstance(leaving_date, datetime):
        ld = leaving_date.date()
    else:
        ld = leaving_date  # may be a date or None

    # Year Group mapping (adjust as needed)
    year_group_mapping = {
        -4: "E1", -3: "E2", -2: "N1", -1: "N2",
         0: "R", 1: "1", 2: "2", 3: "3", 4: "4", 5: "5", 6: "6",
         7: "7", 8: "8", 9: "9", 10: "10", 11: "11",
        12: "12", 13: "13", 14: "14"
    }

    rows = []
    for i in range(years_back):
        # Academic year start rolling backward from the current start year
        ay_start_year = start_year - i
        ay_label = f"{ay_start_year}/{ay_start_year + 1}"
        ay_start_date = date(ay_start_year, 8, 1)

        # Decrement Year Group relative to the CURRENT academic year
        adjusted_year_group = Current_YG - i
        std_year_group_str = year_group_mapping.get(adjusted_year_group, "Unknown")

        rows.append((
            ay_label,
            f"Year {adjusted_year_group}",
            std_year_group_str,
            ay_start_date  # keep for filtering, drop before returning
        ))

    # Filter: remove rows where AY start is AFTER the leaving date
    if ld is not None:
        rows = [r for r in rows if r[3] <= ld]

    # Strip the helper ay_start_date before returning
    return [(r[0], r[1], r[2]) for r in rows]


In [ ]:
# UDF to return std_year_group or fall back to a calculation based on dob
def calculate_age_or_year_group(dob, academic_year, std_Year_Group):
    if std_Year_Group is not None:
        # Use the value of Year_Group
        return std_Year_Group
    elif dob is not None:
        # Calculate age based on Date_Of_Birth
        start_year = int(academic_year.split('/')[0])
        aug_31 = datetime(start_year, 8, 31)
        age = aug_31.year - dob.year - ((aug_31.month, aug_31.day) < (dob.month, dob.day))
        # subtract 4 to adjust for starting year group
        return (age - 4)
    else:
        # Return None if both Date_Of_Birth and Year_Group are missing
        return None


In [ ]:
# UDF for age
calculate_age_udf = F.udf(calculate_age_or_year_group, IntegerType())

# UDF for year group mapping
year_group_udf = F.udf(get_year_group_from_age, IntegerType())

# UDF for generating prior academic years and Year Groups
generate_prior_years_udf = F.udf(generate_academic_years_and_groups, ArrayType(StructType([
    StructField("AcademicYear", StringType(), True),
    StructField("AdjustedYearGroup", StringType(), True),
    StructField("std_year_group", StringType(), True)
])))

In [ ]:
oeai.log.end_block()
oeai.log.checkpoint("Init")

# Processing

##### Create dim_StudentAcademicYearGroup.  Hosts each academic year entry for the student along with their year group.  This is done before the main processing as the StudentAcademicYearGroupkey will be added to the fact tables

In [ ]:
oeai.log.start_block("Processing")

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

try:
    _fn ="dim_StudentAcademicYearGroup - Create"

    # Load dim_Student from the silver path in Delta format
    dim_student = spark.read.format("delta").load(f"{silver_path}/dim_Student")

    # Load dim_StudentExtended from the silver path in Delta format
    dim_studentex = spark.read.format("delta").load(f"{silver_path}/dim_StudentExtended")

    # Load dim_NCYearGroup from reference folder
    reference_path = reference_path + "dim_NCYearGroup.csv"
    dim_NCYearGroup = spark.read.csv(reference_path, header=True, inferSchema=True)

    # Join dim_student with dim_studentex on studentkey
    dim_student = dim_student.join(
        dim_studentex.select("studentkey", "Admission_Date", "Leaving_Date", "Year_Group"),
        on="studentkey",
        how="left"
    )

    # Join dim_student with dim_NCYearGroup on Year_Group_Description
    dim_student = dim_student.join(
        dim_NCYearGroup,
        dim_student["Year_Group"] == dim_NCYearGroup["Year_Group_Description"],
        "left"
    )

    # Select necessary columns
    dim_student_selected = dim_student.select(
        F.col("studentkey"),
        F.col("school_id"),
        F.to_date(F.col("Date_Of_Birth"), "yyyy-MM-dd").alias("Date_Of_Birth"),
        F.to_date(F.col("Admission_Date"), "yyyy-MM-dd").cast(DateType()).alias("Admission_Date"),
        F.to_date(F.col("Leaving_Date"), "yyyy-MM-dd").cast(DateType()).alias("Leaving_Date"),
        F.col("std_Year_Group").cast(IntegerType()).alias("std_Year_Group")
    )


    from pyspark.sql import functions as F
    # Add today's date as a temp column, then derive Academic_Year from it
    dim_student_with_academic_year = (
        ensure_academic_year_column(
            df=dim_student_selected.withColumn("today", F.current_date()),
            date_column="today"
        )
        .drop("today")  # clean up temp col
        .withColumnRenamed("Academic_Year", "Current_Academic_Year")  # rename manually
    )

    # Calculate age-based std year group, or fall back to dob
    dim_student_with_age = dim_student_with_academic_year.withColumn(
        "Current_YG", calculate_age_udf(F.col("Date_Of_Birth"), F.col("Current_Academic_Year"), F.col("std_Year_Group"))
    )

    # Generate prior years and year groups based on age, admission date, and leaving date
    expanded_df = dim_student_with_age.withColumn(
        "Prior_Years",
        generate_prior_years_udf(
            F.col("Current_YG"),
            F.col("Current_Academic_Year"),
            F.col("Leaving_Date")
        )
    )

    # Drop Admission_Date, and Leaving_Date before proceeding to final output
    expanded_df = expanded_df.drop("Admission_Date", "Leaving_Date")

    # Explode the Prior_Years column to get a row for each academic year
    expanded_df = expanded_df.withColumn("Prior_Year", F.explode(F.col("Prior_Years")))
    # Add Compulsory_School_Age_Calc

    expanded_df = expanded_df.withColumn(
        "Ref_Date",
        F.to_date(F.concat(F.split(F.col("Current_Academic_Year"), "/")[0], F.lit("-08-31")))
    )

    expanded_df = expanded_df.withColumn(
        "Age",
        F.year(F.col("Ref_Date")) - F.year(F.col("Date_of_Birth")) -
        F.when(
            F.date_format(F.col("Ref_Date"), "MM-dd") < F.date_format(F.col("Date_of_Birth"), "MM-dd"),
            1
        ).otherwise(0)
    )

    expanded_df = expanded_df.withColumn(
        "CompulsorySchoolAge",
        F.when((F.col("Age") >= 5) & (F.col("Age") <= 15), 1).otherwise(0)
    ).drop("Ref_Date", "Age")

    #Exclude unnecessary columns from the final output
    final_df = expanded_df.select(
        F.col("studentkey"),
        F.col("school_id"),
        F.col("Prior_Year.AcademicYear").alias("Academic_Year"),
        F.col("Prior_Year.AdjustedYearGroup").alias("Year_Group"),
        F.col("Prior_Year.std_year_group").alias("std_year_group"),
        F.col("CompulsorySchoolAge").alias("CompulsorySchoolAge")
    )

    # Add a monotonically increasing ID and adjust it to start from 1
    final_df_with_key = final_df.withColumn(
        "studentacademicyeargroupkey",
        F.row_number().over(Window.orderBy(F.monotonically_increasing_id()))
    )
except:
    oeai.log.exception(f"ERROR - {_fn}", function=_fn)
else:
    oeai.log.debug(f"SUCCESS - {_fn}", function=_fn)

### Filter dim_StudentAcademicYearGroup to exclude students without an attendance record in that academic year
try:
    _fn ="dim_StudentAcademicYearGroup - Filter"

    # Load AttendanceSession from the silver path in Delta format
    df_as = spark.read.format("delta").load(f"{silver_path}/fact_AttendanceSession")
    from pyspark.sql.functions import col, to_date, lit, concat, substring, date_add, sum as F_sum, when, trim, lower

    # --- Build academic year start and end from Academic_Year ---

    start_month = ACADEMIC_YEAR_START.get("month", 8)
    start_day   = ACADEMIC_YEAR_START.get("day", 1)

    final_df_with_key = (
        final_df_with_key
        .withColumn("_base_year", F.substring(F.col("Academic_Year"), 1, 4).cast("int"))
        .withColumn(
            "academic_year_start",
            F.to_date(
                F.concat_ws(
                    "-",
                    F.col("_base_year").cast("string"),
                    F.lpad(F.lit(start_month).cast("string"), 2, "0"),
                    F.lpad(F.lit(start_day).cast("string"), 2, "0"),
                ),
                "yyyy-MM-dd",
            ),
        )
        # academic year end = start + 364 days
        .withColumn("academic_year_end", F.date_add(F.col("academic_year_start"), 364))
        .drop("_base_year")
    )

    # Step 2: Filter AttendanceSession by academic year range
    df_as_filtered_by_year = df_as.join(
        final_df_with_key.select("studentkey", "Academic_Year", "academic_year_start", "academic_year_end"),
        on="studentkey",
        how="inner"
    ).filter(
        (col("Date") >= col("academic_year_start")) &  # Date falls in academic year
        (col("Date") <= col("academic_year_end"))  # Date falls in academic year
    )

    # Step 3: Identify students who only have irrelevant marks (#, Z) or missing marks (-, #, Z)
    students_to_exclude = df_as_filtered_by_year.groupBy("studentkey", "Academic_Year").agg(
        F_sum(when(col("Mark").isin("#", "Z", "-"), 1).otherwise(0)).alias("sum_irrelevant_marks"),
        F_sum(when(~col("Mark").isin("#", "Z", "-"), 1).otherwise(0)).alias("sum_valid_marks")
    ).filter(
        (col("sum_irrelevant_marks") > 0) &  # At least one irrelevant or missing mark
        (col("sum_valid_marks") == 0)  # No valid marks at all
    ).select("studentkey", "Academic_Year")

    # Step 4: Exclude students identified in Step 3
    df_as_valid = df_as_filtered_by_year.join(
        students_to_exclude, on=["studentkey", "Academic_Year"], how="leftanti"
    )

    # Step 5: Drop duplicates based on studentkey and academic year to ensure unique rows
    filtered_df = df_as_valid.dropDuplicates(["studentkey", "Academic_Year"])

    # Step 6: Join back to final_df_with_key to retain valid records only
    result_df = final_df_with_key.join(
        filtered_df.select("studentkey", "Academic_Year").distinct(),
        on=["studentkey", "Academic_Year"],
        how="inner"
    )

    # Step 7: Drop the academic year start and end columns
    result_df_cleaned = result_df.drop("academic_year_start", "academic_year_end")

except:
    oeai.log.exception(f"ERROR - {_fn}", function=_fn)
else:
    oeai.log.debug(f"SUCCESS - {_fn}", function=_fn)

# OnRollOnCensus
try:
    _fn ="dim_StudentAcademicYearGroup - Add OnRollOnCensus"
    # Assuming final_path is already defined
    path = f"{silver_path}/dim_StudentExtended"

    # Read the Delta table
    df_se = spark.read.format("delta").load(path)

    # Assuming dim_student and result_df_cleaned are PySpark DataFrames

    # Perform the join
    result_df_cleaned = result_df_cleaned.join(
        df_se.select("studentkey", "Admission_Date", "Leaving_Date"), 
        on="studentkey", 
        how="left"
    )

    from pyspark.sql import functions as F
    from pyspark.sql.types import DateType
    import calendar
    from datetime import datetime, timedelta

    # Function to calculate the 3rd Thursday in January for a given year
    def third_thursday(year):
        # January 1st of the year
        january_first = datetime(year, 1, 1)
        # Get the weekday of January 1st (0 = Monday, 1 = Tuesday, ..., 6 = Sunday)
        weekday_of_jan_first = january_first.weekday()
        # Calculate the date of the 3rd Thursday
        days_to_thursday = (3 - weekday_of_jan_first) % 7  # Find the first Thursday
        third_thursday_date = january_first + timedelta(days=days_to_thursday + 14)  # Add 14 more days to get the 3rd Thursday
        return third_thursday_date.date()  # Return the date in YYYY-MM-DD format

    # Register the UDF
    def third_thursday_udf(academic_year):
        year = int(academic_year.split('/')[1])  # Extract year after "/"
        return third_thursday(year)

    # Register the function as a UDF
    third_thursday_udf_spark = F.udf(third_thursday_udf, DateType())

    # Add census date
    result_df_cleaned = result_df_cleaned.withColumn("Census_Date", third_thursday_udf_spark(F.col("Academic_Year")))

    # Show the updated DataFrame
    #result_df_cleaned.show(truncate=False)

    from pyspark.sql import functions as F

    # Add OnRollOnCensus

    result_df_cleaned = result_df_cleaned.withColumn(
        "OnRollOnCensus",
        F.when(
            (F.col("Admission_Date").isNull()), 
            "Unknown"
        ).when(
            (F.col("Admission_Date") <= F.col("Census_Date")) & 
            ((F.col("Leaving_Date") >= F.col("Census_Date")) | F.col("Leaving_Date").isNull()),
            1
        ).otherwise(0)
    )
    # Remove unnecessary columns
    result_df_cleaned_dropped = result_df_cleaned.drop("Admission_Date", "Leaving_Date", "Census_Date")

except:
    oeai.log.exception(f"ERROR - {_fn}", function=_fn)
else:
    oeai.log.debug(f"SUCCESS - {_fn}", function=_fn)

# Write the result to a new parquet file
try:
    _fn ="dim_StudentAcademicYearGroup - Write"
    final_path = f"{gold_path}/dim_StudentAcademicYearGroup"
    result_df_cleaned_dropped.write.mode("overwrite").parquet(final_path)

except:
    oeai.log.exception(f"ERROR - {_fn}", function=_fn)
else:
    oeai.log.info(f"SUCCESS - {_fn}", function=_fn)

oeai.log.end_block()
oeai.log.checkpoint("Processing")

# Calculated Gold tables

In [ ]:
oeai.log.start_block("Calculated Gold tables")

##### Create dim_StudentCalculated. This section is to create a dim_StudentCalculated to host a number of calculated measures relevant to analytics, including:
       1. is_sen and Is_SEN_String (SEN or Not SEN)
       2. is_summerborn and Is_Summer_Born_String (Summer Born or Not Summer Born)
       3. is_current and Is_Current_String (Current or Not Current)
       4. Full name field (concat sur, for)
       5. PP eligable string
       6. EAL string

In [ ]:
oeai.log.start_block("dim_StudentCalculated")

# Define the schema for the dim_StudentCalculated table
dim_student_calculated_schema = StructType([
    StructField("studentkey", StringType(), True),
    StructField("is_sen", IntegerType(), True),
    StructField("Is_SEN_String", StringType(), True),
    StructField("Is_PP_String", StringType(), True),
    StructField("Is_EAL_String", StringType(), True),
    StructField("is_summer_born", IntegerType(), True),
    StructField("Is_Summer_Born_String", StringType(), True),
    #StructField("is_current", IntegerType(), True),
    StructField("Is_Current_String", StringType(), True),
    StructField("Full_Name", StringType(), True),
    #StructField("is_in_year_joiner", IntegerType(), True),
    #StructField("Is_In_Year_Joiner_String", StringType(), True),
])

try:
    _fn ="dim_StudentCalculated - Load"

    # Load the student and studentextended tables as Delta from silver_path
    silver_student_dir = '/dim_Student'
    silver_student_path = f"{silver_path}{silver_student_dir}/"
    dim_student_df = (
        spark
        .read
        .format("delta")
        .load(silver_student_path)
    )

    silver_studentextended_dir = '/dim_StudentExtended'
    silver_studentextended_path = f"{silver_path}{silver_studentextended_dir}/"
    dim_studentextended_df = (
        spark
        .read
        .format("delta")
        .load(silver_studentextended_path)
    )
except:
    oeai.log.exception(f"ERROR - {_fn}", function=_fn)
else:
    oeai.log.debug(f"SUCCESS - {_fn}", function=_fn)

try:
    _fn ="dim_StudentCalculated - Process"
    # Calculate the is_summer_born and Is_Summer_Born_String columns
    dim_student_df = dim_student_df.withColumn(
        "is_summer_born",
        when(
            (month(col("Date_Of_Birth")).between(4, 8)) &
            ((month(col("Date_Of_Birth")) > 4) | (month(col("Date_Of_Birth")) == 4) & (dayofmonth(col("Date_Of_Birth")) >= 1)) &
            ((month(col("Date_Of_Birth")) < 8) | (month(col("Date_Of_Birth")) == 8) & (dayofmonth(col("Date_Of_Birth")) <= 31)),
            1
        ).otherwise(0)
    ).withColumn(
        "Is_Summer_Born_String",
        when(col("is_summer_born") == 1, "Summer Born").otherwise("Not Summer Born")
    )

    # Add is_sen, Is_SEN_String, is_current, and Is_Current_String based on dim_studentextended_df
    dim_studentextended_df = dim_studentextended_df.withColumn(
        "is_sen",
        when(col("SEN_Status").isin("K", "E"), 1).otherwise(0)
    ).withColumn(
        "Is_SEN_String",
        when(col("SEN_Status").isin("K", "E"), "SEN").otherwise("Not SEN")
    ).withColumn(
        "Is_PP_String",
        when(col("Pupil_Premium_Indicator") == True, "PP").otherwise("Not PP")
    ).withColumn(
        "Is_EAL_String",
        when(col("English_As_Additional_Language") == True, "EAL").otherwise("Not EAL")
    ).withColumn(
        "Is_Current_String",
        when(col("Is_Current") == True, "On Roll").otherwise("Leaver")
    )

    # Add Full_Name
    dim_student_df = dim_student_df.withColumn(
        "Full_Name",
        F.concat(
            F.col("Legal_Surname"),
            F.lit(", "),  # Adds a comma and a space between the surname and forename
            F.col("Legal_Forename")
        )
    )

    # Join dim_student_df with dim_studentextended_df
    dim_student_calculated_df = dim_student_df.join(dim_studentextended_df, "studentkey", "left").select(
        col("studentkey"),
        col("is_sen"),
        col("Is_SEN_String"),
        col("Is_PP_String"),
        col("Is_EAL_String"),
        col("is_summer_born"),
        col("Is_Summer_Born_String"),
        #col("is_current"),
        col("Is_Current_String"),
        col("Full_Name"),
        #col("is_in_year_joiner"),
        #col("Is_In_Year_Joiner_String")
    )

    # Ensure that the DataFrame conforms to the schema
    dim_student_calculated_df = spark.createDataFrame(dim_student_calculated_df.rdd, dim_student_calculated_schema)

except:
    oeai.log.exception(f"ERROR - {_fn}", function=_fn)
else:
    oeai.log.debug(f"SUCCESS - {_fn}", function=_fn)

# Write the result to a new parquet file
try:
    _fn ="dim_StudentCalculated - Write"
    dim_student_calculated_df.write.mode("overwrite").format("parquet").save(gold_path + "/dim_StudentCalculated")
except:
    oeai.log.exception(f"ERROR - {_fn}", function=_fn)
else:
    oeai.log.info(f"SUCCESS - {_fn}", function=_fn)

oeai.log.end_block()
oeai.log.checkpoint("dim_StudentCalculated")

##### fact_AttendanceSummary

In [ ]:
oeai.log.start_block("fact_AttendanceSummary")

try:
    _fn ="fact_AttendanceSummary"

    from pyspark.sql.functions import sum as _sum, concat_ws, col, lit, when, collect_list, row_number, min, max, first, date_format, to_date
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType
    from pyspark.sql import Window

    fact_AttendanceSession = spark.read.format("delta").load(silver_path + 'fact_AttendanceSession')

    fact_AttendanceSession = ensure_academic_year_column(fact_AttendanceSession, "Date")

    # ---- Helper key: unique_student_id (student_id + school_id) ---------------
    # Use concat_ws to handle potential nulls gracefully (won't produce 'null_')
    fact_AttendanceSession = fact_AttendanceSession.withColumn(
        "unique_student_id", F.concat_ws("_", F.col("student_id"), F.col("school_id"))
    )


    # Cast boolean columns to integer before summing
    fact_AttendanceSession = fact_AttendanceSession.withColumn("is_present", col("is_present").cast("int")) \
                                                .withColumn("is_auth_abs", col("is_auth_abs").cast("int")) \
                                                .withColumn("is_unauth_abs", col("is_unauth_abs").cast("int")) \
                                                .withColumn("is_possible", col("is_possible").cast("int")) \
                                                .withColumn("is_attend", col("is_attend").cast("int")) \
                                                .withColumn("is_aea", col("is_aea").cast("int"))

    # Aggregating the fact_AttendanceSession dataframe by unique_student_id and AcademicYear
    aggregated_df = fact_AttendanceSession.groupBy("unique_student_id", "Academic_Year").agg(
        F.sum("is_present").alias("Present"),
        F.sum("is_auth_abs").alias("Authorised_Absences"),
        F.sum("is_unauth_abs").alias("Unauthorised_Absences"),
        F.sum("is_possible").alias("Possible_Marks"),
        F.sum("is_aea").alias("Approved_Educational_Activity"),
        F.sum(F.when(F.col("Mark") == "L", 1).otherwise(0)).alias("Late_Before_Registration"),
        F.sum(F.when(F.col("Mark") == "U", 1).otherwise(0)).alias("Late_After_Registration"),
        F.first("unique_key").alias("unique_key"),
        F.first("student_id").alias("student_id"),
        F.first("studentkey").alias("studentkey"),
        F.first("organisationkey").alias("organisationkey"),
        F.first("school_id").alias("school_id")
    )

    # Aggregation step 2: Concatenating Mark in date order
    window_spec = Window.partitionBy("unique_student_id", "Academic_Year").orderBy("Date", "Session")
    sorted_marks_df = fact_AttendanceSession.withColumn("sorted_mark", col("Mark")).withColumn("rn", row_number().over(window_spec))
    concatenated_marks_df = sorted_marks_df.groupBy("unique_student_id", "Academic_Year").agg(
        concat_ws("", collect_list("sorted_mark")).alias("Attendance_Mark_String")
    )

    concatenated_marks_df = concatenated_marks_df.withColumnRenamed("unique_student_id", "marks_unique_student_id")
    concatenated_marks_df = concatenated_marks_df.withColumnRenamed("Academic_Year", "marks_Academic_Year")

    # Joining aggregated data with concatenated marks
    summary_df = aggregated_df.join(concatenated_marks_df, 
                                    (aggregated_df["unique_student_id"] == concatenated_marks_df["marks_unique_student_id"]) & 
                                    (aggregated_df["Academic_Year"] == concatenated_marks_df["marks_Academic_Year"]),
                                    "left")

    summary_df = summary_df.drop(concatenated_marks_df["marks_unique_student_id"])
    summary_df = summary_df.drop(concatenated_marks_df["marks_Academic_Year"])


# Adding other necessary columns
    summary_df = summary_df.withColumn("Percentage_Attendance", ((col("Present") + col("Approved_Educational_Activity")) / col("Possible_Marks")).cast("decimal(10,4)")) \
                        .withColumn("Percentage_Authorised_Absence", (col("Authorised_Absences") / col("Possible_Marks")).cast("decimal(10,4)")) \
                        .withColumn("Percentage_Unauthorised_Absence", (col("Unauthorised_Absences") / col("Possible_Marks")).cast("decimal(10,4)")) \
                        .withColumn("external_id", lit(None).cast(StringType())) \
                        .withColumn("attendancesummarykey", lit(None).cast(StringType())) \
                        .withColumn("last_updated", lit(None).cast(StringType())) \
                        .withColumn("Unexplained_Absences", lit(None).cast(IntegerType())) \
                        .withColumn("Missing_Marks", lit(None).cast(IntegerType())) \
                        .withColumn("Percentage_Missing", lit(None).cast("decimal(10,4)")) \
                        .withColumn("Attendance_Not_Required", lit(None).cast(IntegerType())) \
                        .withColumn("Percentage_Unexplained_Absence", lit(None).cast(DecimalType(10, 0))) \
                        .withColumn(
                            "Is_Persistently_Absent",
                            when((col("Percentage_Attendance") <= 0.9) & (col("Possible_Marks") >= 20), 1).otherwise(0).cast(IntegerType())
                        ) \
                        .withColumn(
                            "Is_Severely_Absent",
                            when((col("Percentage_Attendance") <= 0.5) & (col("Possible_Marks") >= 20), 1).otherwise(0).cast(IntegerType())
                        )

    #  attendance bands
    df_attendance_summary = summary_df
    df_attendance_summary = df_attendance_summary.withColumn("under_50",when(col("Percentage_Attendance") <= 0.5, 1).otherwise(0)) 
    df_attendance_summary = df_attendance_summary.withColumn("50_to_70",when((col("Percentage_Attendance") > 0.5) & (col("Percentage_Attendance") <= 0.7), 1).otherwise(0))
    df_attendance_summary = df_attendance_summary.withColumn("70_to_80",when((col("Percentage_Attendance") > 0.7) & (col("Percentage_Attendance") <= 0.8), 1).otherwise(0))
    df_attendance_summary = df_attendance_summary.withColumn("80_to_90",when((col("Percentage_Attendance") > 0.8) & (col("Percentage_Attendance") <= 0.9), 1).otherwise(0))
    df_attendance_summary = df_attendance_summary.withColumn("90_to_92",when((col("Percentage_Attendance") > 0.9) & (col("Percentage_Attendance") <= 0.92), 1).otherwise(0))
    df_attendance_summary = df_attendance_summary.withColumn("92_to_95",when((col("Percentage_Attendance") > 0.92) & (col("Percentage_Attendance") <= 0.95), 1).otherwise(0))
    df_attendance_summary = df_attendance_summary.withColumn("95_to_98",when((col("Percentage_Attendance") > 0.95) & (col("Percentage_Attendance") <= 0.98), 1).otherwise(0))
    df_attendance_summary = df_attendance_summary.withColumn("above_98",when(col("Percentage_Attendance") > 0.98, 1).otherwise(0))             

    # Create a single 'Attendance_Bin' column
    df_attendance_summary = df_attendance_summary.withColumn(
        "Attendance_Bin",
        when(col("Percentage_Attendance") <= 0.5, "under 50%")
        .when((col("Percentage_Attendance") > 0.5) & (col("Percentage_Attendance") <= 0.7), "50% to 70%")
        .when((col("Percentage_Attendance") > 0.7) & (col("Percentage_Attendance") <= 0.8), "70% to 80%")
        .when((col("Percentage_Attendance") > 0.8) & (col("Percentage_Attendance") <= 0.9), "80% to 90%")
        .when((col("Percentage_Attendance") > 0.9) & (col("Percentage_Attendance") <= 0.92), "90% to 92%")
        .when((col("Percentage_Attendance") > 0.92) & (col("Percentage_Attendance") <= 0.95), "92% to 95%")
        .when((col("Percentage_Attendance") > 0.95) & (col("Percentage_Attendance") <= 0.98), "95% to 98%")
        .when(col("Percentage_Attendance") > 0.98, "above 98%")
        .otherwise("Unspecified")  # This is optional and can handle any data outside the expected ranges
    )

    # Populate fact_AttendanceSummary
    fact_AttendanceSummary = df_attendance_summary

    #  add the unique_key column to the AttendanceSession dataframe
    fact_AttendanceSummary = fact_AttendanceSummary.withColumn("unique_key", concat(col("unique_student_id"), col("Academic_Year")))

    deltaTablePath_AttSum = silver_path + "fact_AttendanceSummary"
    fact_AttendanceSummary.write.format("delta").mode("overwrite").save(deltaTablePath_AttSum)
except Exception as e:
    oeai.log.exception(f"ERROR - {_fn}", function=_fn)
    # print(f"Error processing table {table_name}: {e}")

else:
    oeai.log.info(f"SUCCESS - {_fn}", function=_fn)

oeai.log.end_block()
oeai.log.checkpoint("fact_AttendanceSummary")

##### Main processing block.  Iterates through each delta table to optimise, enrich and produce the parquet in the Gold layer.

In [ ]:
oeai.log.start_block("Main processing block")

from datetime import datetime
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.functions import col
from pyspark.sql import Window
from pyspark.sql.functions import desc, row_number

# Fact tables processing loop with debugging
_fn ="Main processing block"
for table_name in delta_tables:
    silver_path_delta = f"{silver_path}/{table_name}"
    gold_path_delta = f"{gold_path}/{table_name}"
    
    try:
        delta_table = DeltaTable.forPath(spark, silver_path_delta)
        df = delta_table.toDF()
        initial_count = df.count()
        
        # Apply transformations based on table
        if table_name == "fact_AttendanceSession":
            # --------------------------------------------------------------------------
            # 1) Rename, ensure schema adjustments, filter
            # --------------------------------------------------------------------------
            df = df.drop("external_id", "unique_student_id", "student_id", "attendancesessionkey",
                         "Comment", "Employee", "last_updated", "unique_key", "organisationkey")
            df = ensure_academic_year_column(df, "Date")
            df = add_student_academic_year_group_key(df)
            
            df = df.filter(col("studentacademicyeargroupkey").cast("int").isNotNull())

            # Cast to int
            df = (df
                .withColumn("is_present", F.col("is_present").cast("int"))
                .withColumn("is_auth_abs", F.col("is_auth_abs").cast("int"))
                .withColumn("is_unauth_abs", F.col("is_unauth_abs").cast("int"))
                .withColumn("is_possible", F.col("is_possible").cast("int"))
                .withColumn("is_aea", F.col("is_aea").cast("int"))
                .withColumn("is_attend", F.col("is_attend").cast("int"))
                .withColumn("is_nr", F.col("is_nr").cast("int"))
                .withColumn("is_late_L", F.col("is_late_L").cast("int"))
                .withColumn("is_late_U", F.col("is_late_U").cast("int"))
                .withColumn("is_missing", F.col("is_missing").cast("int"))
            )   
        

            
            # --------------------------------------------------------------------------
            # 2) Convert Date to a proper date type for window ops
            # --------------------------------------------------------------------------
            df = df.withColumn("Date_as_date", F.to_date(F.col("Date"), "dd/MM/yyyy"))
            # Drop rows where Date_as_date > today
            df = df.filter(F.col("Date_as_date") <= F.current_date())  
        
            # --------------------------------------------------------------------------
            # 3) Rolling 10-week logic for 'Is_10_in_10'
            #    - For each session row, sum is_unauth_abs across the last 10 *active* weeks
            #    - A week is "active" if it has one or more 'is_possible' marks for that student
            #    - If that sum >= 10 at this row, flag = 1
            # --------------------------------------------------------------------------
            # 3a) Define a "week_start" column to group sessions into weeks
            df = df.withColumn("week_start", F.date_trunc("week", F.col("Date_as_date")))
        
            # 3b) Summarise unauthorised absences and possible marks at week level
            week_df = (
                df.groupBy("studentacademicyeargroupkey", "Academic_Year", "week_start")
                .agg(
                    F.sum("is_unauth_abs").alias("unauth_abs_per_week"),
                    F.sum("is_possible").alias("possible_per_week")
                )
                # Keep only active weeks
                .filter(F.col("possible_per_week") > 0)
            )
        
            # 3c) Define a rolling window of the last 10 active weeks
            week_window_10 = (
                Window
                .partitionBy("studentacademicyeargroupkey", "Academic_Year")
                .orderBy("week_start")
                .rowsBetween(-9, 0)  # includes current week + 9 preceding active weeks
            )
        
            # 3d) Rolling sum of unauthorised absences across those 10 active weeks
            week_df = week_df.withColumn(
                "rolling_10week_unauth_abs",
                F.sum("unauth_abs_per_week").over(week_window_10)
            )
        
            # 3e) Flag: if rolling_10week_unauth_abs >= 10 => Is_10_in_10_week = 1
            week_df = week_df.withColumn(
                "Is_10_in_10_week",
                F.when(F.col("rolling_10week_unauth_abs") >= 10, 1).otherwise(0)
            )
        
            # 3f) Bring the week-level flag back to session-level
            df = (
                df.alias("sess")
                .join(
                    week_df.select(
                        "studentacademicyeargroupkey", 
                        "Academic_Year", 
                        "week_start", 
                        "Is_10_in_10_week"
                    ).alias("wk"),
                    on=["studentacademicyeargroupkey", "Academic_Year", "week_start"],
                    how="left"
                )
                .withColumn("Is_10_in_10", F.col("wk.Is_10_in_10_week"))
                .drop("wk.Is_10_in_10_week")
            )
            # 3g) Fill forward the last known Is_10_in_10 value across sessions
            fill_window = (
                Window
                .partitionBy("studentacademicyeargroupkey", "Academic_Year")
                .orderBy("Date_as_date", "Session")
                .rowsBetween(Window.unboundedPreceding, 0)
            )

            df = df.withColumn(
                "Is_10_in_10_filled",
                F.last("Is_10_in_10", ignorenulls=True).over(fill_window)
            ).drop("Is_10_in_10") \
            .withColumnRenamed("Is_10_in_10_filled", "Is_10_in_10")
        
            # --------------------------------------------------------------------------
            # 4) 3-day consecutive absence -> '3_days_abs' = 1 if this session's day
            #    is part of at least 3 consecutive absent days.
            # --------------------------------------------------------------------------
            daily_abs_df = (df
                .groupBy("studentacademicyeargroupkey", "Academic_Year", "Date_as_date")
                .agg(F.sum(F.col("is_auth_abs") + F.col("is_unauth_abs")).alias("absences_that_day"))
                .withColumn("is_abs_day", F.when(F.col("absences_that_day") > 0, 1).otherwise(0))
            )
        
            day_window = (
                Window
                .partitionBy("studentacademicyeargroupkey", "Academic_Year")
                .orderBy("Date_as_date")
            )
            daily_abs_df = (daily_abs_df
                .withColumn("prev1", F.lag("is_abs_day", 1).over(day_window))
                .withColumn("prev2", F.lag("is_abs_day", 2).over(day_window))
                .withColumn(
                    "has_3consecutive",
                    F.when(
                        (F.col("is_abs_day") == 1) &
                        (F.col("prev1") == 1) &
                        (F.col("prev2") == 1),
                        1
                    ).otherwise(0)
                )
            )
        
            daily_abs_df = daily_abs_df.select(
                "studentacademicyeargroupkey", "Academic_Year", "Date_as_date", "has_3consecutive"
            )
            df = (df
                .join(
                    daily_abs_df,
                    on=["studentacademicyeargroupkey", "Academic_Year", "Date_as_date"],
                    how="left"
                )
                .withColumn("3_days_abs", F.col("has_3consecutive"))
                .drop("has_3consecutive")
            )
        
            # --------------------------------------------------------------------------
            # 5) abs_after_exclusion -> if the PREVIOUS session Mark == 'E' and
            #    CURRENT session Mark in [C,E,G,H,I,M,N,O,R,S,T,U,C1,C2,J1]
            # --------------------------------------------------------------------------
            absence_codes = ["C","G","H","I","M","N","O","R","S","T","U","C1","C2","J1"]
        
            session_window_unbounded = (
                Window
                .partitionBy("studentacademicyeargroupkey", "Academic_Year")
                .orderBy("Date_as_date", "Session")
                .rowsBetween(Window.unboundedPreceding, 0)
            )
        
            df = df.withColumn(
                "exclusion_count",
                F.sum(F.when(F.col("Mark") == "E", 1).otherwise(0)).over(session_window_unbounded)
            )
        
            df = df.withColumn(
                "non_absence_count",
                F.sum(
                    F.when(
                        ~F.col("Mark").isin(absence_codes + ["E"]), 1
                    ).otherwise(0)
                ).over(session_window_unbounded)
            )
        
            df = df.withColumn(
                "abs_after_exclusion",
                F.when(
                    (F.col("exclusion_count") > F.col("non_absence_count"))
            & (F.col("Mark").isin(absence_codes)),
                    1
                ).otherwise(0)
            )
        
            df = df.drop("exclusion_count", "non_absence_count")
            # --------------------------------------------------------------------------
            # 6) Define a Window to accumulate from the start of each academic year
            # --------------------------------------------------------------------------
            window_spec = (
                Window
                .partitionBy("studentacademicyeargroupkey", "Academic_Year")
                .orderBy(F.col("Date").asc())
                .rowsBetween(Window.unboundedPreceding, Window.currentRow)
            )
            # --------------------------------------------------------------------------
            # 7) Compute cumulative sums, derive attendance ratio
            # --------------------------------------------------------------------------
            df = (df
                .withColumn("cumulative_present", F.sum("is_attend").over(window_spec))
                .withColumn("cumulative_possible", F.sum("is_possible").over(window_spec))
                .withColumn(
                    "attendance_ratio",
                    F.when(
                        F.col("cumulative_possible") > 0,
                        F.col("cumulative_present") / F.col("cumulative_possible")
                    )
                )
            )
        
            # --------------------------------------------------------------------------
            # 8) Label each row with is_PA / is_SA
            #    - is_PA = 1 if attendance_ratio <= 0.90 and sum_possible > 20
            #    - is_SA = 1 if attendance_ratio <= 0.50 and sum_possible > 20
            # --------------------------------------------------------------------------
            df = (df
                .withColumn(
                    "is_PA",
                    F.when(
                        (F.col("attendance_ratio") <= 0.90) & (F.col("cumulative_possible") > 20),
                        1
                    ).otherwise(0)
                )
                .withColumn(
                    "is_SA",
                    F.when(
                        (F.col("attendance_ratio") <= 0.50) & (F.col("cumulative_possible") > 20),
                        1
                    ).otherwise(0)
                )
            )
            # --------------------------------------------------------------------------
            # Final cleanup
            # --------------------------------------------------------------------------
            df = df.drop("Date_as_date", "week_start", "rolling_10week_unauth_abs", "Is_10_in_10_week")

        elif table_name == "fact_Behaviour":
            df = df.drop("behaviourkey", "external_id", "unique_student_id", "unique_key", "student_id", 
                         "organisationkey", "Location", "Status", "Comment", "Subject", "Class", "Total_Points", "last_updated", "Is_Deleted")
            df = df.withColumn("Points", F.abs(F.col("Points")))  # Make Points absolute
            df = ensure_academic_year_column(df, "Incident_Date")
            df = add_student_academic_year_group_key(df)
            
            df = df.filter(col("studentacademicyeargroupkey").cast("int").isNotNull())

        elif table_name == "fact_Achievement":
            df = df.drop("achievementkey", "external_id", "organisationkey", "unique_student_id", "unique_key",
                         "student_id", "action_date", "Subject", "Class", "Total_Points", "Comment", "Parents_Notified", "last_updated", "Is_Deleted")
            df = df.withColumn("Points", F.abs(F.col("Points")))  # Make Points absolute
            df = ensure_academic_year_column(df, "Achievement_Date")
            df = add_student_academic_year_group_key(df)
            
            df = df.filter(col("studentacademicyeargroupkey").cast("int").isNotNull())

        elif table_name == "fact_AttendanceSummary":
            df = add_student_academic_year_group_key(df)
            
        elif table_name == "fact_AttendanceLesson":
            df = df.filter(col("Subject").isNotNull())
            df = ensure_academic_year_column(df, "Start")
            df = add_student_academic_year_group_key(df)
        
        elif table_name == "dim_Student":
            df = df.withColumn(
                "Sex",
                F.when(F.col("Sex") == "M", "Male")
                .when(F.col("Sex") == "F", "Female")
                .otherwise(F.col("Sex"))
            )

        elif table_name == "dim_Staff":
            df = df.withColumn(
                "Sex",
                F.when(F.col("Sex") == "M", "Male")
                .when(F.col("Sex") == "F", "Female")
                .otherwise(F.col("Sex"))
            )     

        elif table_name == "fact_Detention":
            df = ensure_academic_year_column(df, "Start")
            df = add_student_academic_year_group_key(df)
         
        elif table_name == "fact_PointAward":
            df = ensure_academic_year_column(df, "Date")
            df = add_student_academic_year_group_key(df)
             
        elif table_name == "fact_InternalExclusion":
            df = ensure_academic_year_column(df, "Start")
            df = add_student_academic_year_group_key(df)
            
        elif table_name == "fact_Exclusion":
            df = df.drop("Academic_Year")

            df = ensure_academic_year_column(df, "Start_Date")

            df = add_student_academic_year_group_key(df)
           

            df = df.filter(col("studentacademicyeargroupkey").cast("int").isNotNull())
            
            df = df.withColumn("Days", F.when((F.col("Days").isNull()) | (F.col("Days") == "None"), 0).otherwise(F.col("Days")))
            
            # Standardize Exclusion Types
            df = df.withColumn("Type", 
                               F.when(F.col("Type") == "FixedPeriodExclusion", "Suspension")
                                .when(F.col("Type") == "PermanentExclusion", "Permanent")
                                .otherwise(F.col("Type")))
            df = df.withColumn("Type_Code", 
                               F.when(F.col("Type_Code") == "FixedPeriodExclusion", "SUSP")
                                .when(F.col("Type_Code") == "PermanentExclusion", "PERM")
                                .otherwise(F.col("Type_Code")))

            # Drop duplicates based on the specified columns to handle the situation where a school deletes an exclusion and replaces it resulting in a duplicate
            df = df.dropDuplicates(['Academic_Year', 'school_id', 'student_id', 'Start_Date', 'End_Date', 'Days'])

        # Write the DataFrame to Parquet
        # print(f"Table: {table_name} now writing out")
        df.write.format("parquet").mode("overwrite").save(gold_path_delta)
    
    except:
        print(f"Error processing table {table_name}: {e}")
        oeai.log.exception(f"ERROR - {table_name}", function=_fn, table_name=table_name)
    
    else:
        oeai.log.info(f"SUCCESS - {table_name}", function=_fn, table_name=table_name)

oeai.log.end_block()
oeai.log.checkpoint("Main processing block")

In [ ]:
oeai.log.end_block()
oeai.log.checkpoint("Calculated Gold tables")

In [ ]:
oeai.log.end_block(index=0, include_target=True)
oeai.log.shutdown()
oeai.log.checkpoint("shutdown")